In [1]:
import sys
import os
import json
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output
import numpy as np
import pandas as pd
import geopandas as gpd
import shutil
import os
    

sys.path.append('../')
import model.dataloader as dataloader
from model.dataset import LABEL_MAPPING


In [2]:
inverse_label_mapping = {value : key for key, value in LABEL_MAPPING.items()}

In [3]:
_id = 'aew9h7hg'


In [4]:


model_dir = os.path.join('../models_trained', _id)
config_path = os.path.join(model_dir, 'config.json')
with open(config_path) as source:
    config = json.loads(source.read())


In [5]:
config['data']['annotation_file'] =  '/home/jan/Documents/EPFL/project_2/data/labels/dataset_v5/labels_no_regrowth_fixed_country_names.csv'
config['hyperparams']['image_encoder']['use'] = False

In [6]:
# fetch dataloaders
dataloaders = dataloader.fetch_dataloader(
    ['train', 'val'], config)
train_dl = dataloaders['train']
val_dl = dataloaders['val']

In [7]:
x,y = next(iter(train_dl))

In [8]:
x[1].shape

torch.Size([64, 2])

In [9]:
train_dl.dataset.img_labels.iloc[0]

driver                                              cocoa
sampleid                                        170000012
dataset                             cocoa_peru_points.csv
country                                              Peru
geometry    POINT (-75.86782891784124 -6.463607769725906)
block_id                                             67.0
split                                               train
dst_size                                            100.0
Name: 0, dtype: object

In [10]:
device = 'cuda'

In [11]:
all_rates = []
for train_batch, labels_batch in train_dl:

    arr = train_batch[2].numpy()
    #print(arr.shape)
    all_rates.append(arr)
    

In [12]:
train_tab_values = np.concatenate(all_rates)

In [13]:
all_rates = []
for train_batch, labels_batch in val_dl:

    arr = train_batch[2].numpy()
    #print(arr.shape)
    all_rates.append(arr)
    

In [14]:
test_tab_values = np.concatenate(all_rates)

In [15]:
print(train_tab_values.shape)

(10000, 11)


In [16]:
tab_values = np.concatenate([train_tab_values, test_tab_values])

In [17]:
tab_values.shape

(14755, 11)

In [3]:
df = pd.read_csv('../data/ZENODO/labels.csv')


FileNotFoundError: [Errno 2] No such file or directory: '../data/ZENODO/labels.csv'

In [21]:
df = pd.read_csv('../data/ZENODO/labels.csv')
train = df.loc[df.split == 'train']
test = df.loc[df.split == 'test']
df = pd.concat([train, test]).reset_index(drop=True)


In [22]:
LABEL_MAPPING = {
    'pasture' : 0, 
    'forest plantation' : 1,
    'soy' : 2,
    'coffee' : 3,
    'palm oil': 4,
    'mining': 5,
    'urban': 6,
    'cocoa': 7,
    'rubber': 8,
    'shifting agriculture': 9,
    'other': 10 
}

classes = [key for key in LABEL_MAPPING.keys()]

In [23]:
df[classes] = tab_values

In [71]:
df = df.drop(['dataset', 'block_id', 'dst_size'], axis=1)

In [79]:
df['split'] = df['split'].apply(lambda x: 'test' if x == 4 else 'train')

In [82]:
df.to_csv('../data/ZENODO/labels.csv', index=False)

## Copy images and deforestation masks

In [4]:
df = pd.read_csv('../data/PLUTo/labels.csv')

In [5]:

sampleids = df.sampleid.tolist()

In [6]:
import tqdm

In [6]:

img_dir = '/home/jan/Documents/EPFL/project_2/data/images/dataset_v5/four_images/'
forestloss_dir = '/home/jan/Documents/EPFL/project_2/data/forestloss_tiles/dataset_v5/'
img_target_dir = '/home/jan/Documents/EPFL/project_2/data/ZENODO/images/'
forestloss_target_dir = '/home/jan/Documents/EPFL/project_2/data/ZENODO/deforestation_masks/'
for sampleid in tqdm.tqdm(sampleids):
    shutil.copytree(os.path.join(img_dir, str(sampleid)), os.path.join(img_target_dir, str(sampleid)))
    shutil.copy(os.path.join(forestloss_dir, str(sampleid) + ".tif"), forestloss_target_dir)
    

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14755/14755 [03:32<00:00, 69.37it/s]


In [7]:
for sampleid in tqdm.tqdm(sampleids):
    assert len(os.listdir(os.path.join(img_target_dir, str(sampleid)))) == 4

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14755/14755 [00:00<00:00, 65218.93it/s]


In [9]:
df.columns

Index(['driver', 'sampleid', 'country', 'geometry', 'split', 'pasture',
       'forest plantation', 'soy', 'coffee', 'palm oil', 'mining', 'urban',
       'cocoa', 'rubber', 'shifting agriculture', 'other'],
      dtype='object')

In [24]:
tab_data = df[['sampleid', 'pasture',
       'forest plantation', 'soy', 'coffee', 'palm oil', 'mining', 'urban',
       'cocoa', 'rubber', 'shifting agriculture', 'other']]

In [25]:
df = df[['driver', 'sampleid', 'country', 'geometry', 'split']]

In [26]:
tab_data.to_csv('../data/ZENODO/tabular_data.csv', index=False)

In [21]:
df.to_csv('../data/ZENODO/labels.csv', index=False)

In [40]:
df

,driver,sampleid,country,geometry,split
0,cocoa,170000012,Peru,POINT (-75.86782891784124 -6.463607769725906),train
1,cocoa,170000017,Peru,POINT (-75.9478688096563 -6.605900910304538),train
2,cocoa,170000018,Peru,POINT (-76.16130852116306 -6.6236875528769),train
3,cocoa,170000020,Peru,POINT (-76.94392079668802 -6.881593870177058),train
4,cocoa,170000021,Peru,POINT (-76.3569615900443 -6.934953797894544),train
...,...,...,...,...,...
14750,forest plantation,10655614,Brazil,POINT (-42.75638979073803 -19.484727124722763),test
14751,forest plantation,10655563,Brazil,POINT (-42.41844358085226 -19.466940482139698),test
14752,forest plantation,10656797,Brazil,POINT (-43.34334899738171 -19.973859795765435),test
14753,forest plantation,10653072,Brazil,POINT (-42.43623022347783 -18.533141746558393),test


## Final touches



In [28]:
labels = pd.read_csv('/home/jan/Documents/EPFL/project_2/data/labels/dataset_v5/labels_no_regrowth_fixed_country_names.csv')


In [29]:
labels.dataset.value_counts()

dataset
mapbiomas.csv                     6474
mapbiomas_indonesia.csv           3551
rubber.csv                        2983
tyukavina_congo_basin_agri.csv    2821
mines.csv                         2486
worldcover.csv                    2428
palm_oil.csv                      2319
cocoa.csv                         2195
soy_south_america.csv             2137
cocoa_ecuador.csv                  645
mapbiomas_peru.csv                 585
manual_annotation.csv              405
mapbiomas_paraguay.csv             405
colombia_pasture.csv               298
coffee_maskell.csv                 283
mapbiomas_bolivia.csv              237
cocoa_peru_points.csv               44
coffee_sdpt_brazil.csv              41
mapbiomas_venezuela.csv             34
Name: count, dtype: int64

In [30]:
sampleids_to_exclude = labels.loc[labels.dataset.isin(['cocoa_peru_points.csv', 'colombia_pasture.csv'])].sampleid.tolist()


In [31]:
df = pd.read_csv('../data/ZENODO/labels.csv')

In [32]:
filtered_df = df.loc[~df.sampleid.isin(sampleids_to_exclude)].reset_index(drop=True)


In [33]:
filtered_df

,driver,sampleid,country,geometry,split
0,coffee,190000000,Ethiopia,POINT (38.22700943219967 6.191705990191045),train
1,coffee,190000001,Ethiopia,POINT (38.27551845754213 5.951586315467204),train
2,coffee,190000002,Ethiopia,POINT (38.28441177885492 5.951586315467204),train
3,coffee,190000003,Ethiopia,POINT (38.26905058749646 5.942692994181162),train
4,coffee,190000004,Ethiopia,POINT (38.27794390880926 5.942692994181162),train
...,...,...,...,...,...
14627,forest plantation,10655614,Brazil,POINT (-42.75638979073803 -19.484727124722763),test
14628,forest plantation,10655563,Brazil,POINT (-42.41844358085226 -19.466940482139698),test
14629,forest plantation,10656797,Brazil,POINT (-43.34334899738171 -19.973859795765435),test
14630,forest plantation,10653072,Brazil,POINT (-42.43623022347783 -18.533141746558393),test


In [ ]:
filtered_df.to_csv('../data/ZENODO/labels.csv', index=False)

In [34]:
tab = pd.read_csv('../data/ZENODO/tabular_data.csv')

In [36]:
tab = tab.loc[~tab.sampleid.isin(sampleids_to_exclude)].reset_index(drop=True)


In [38]:
tab.to_csv('../data/ZENODO/tabular_data.csv')

In [34]:
import tqdm

In [39]:
xx = df.loc[df.sampleid.isin(sampleids_to_exclude)].reset_index(drop=True).sampleid.tolist()

In [41]:
img_target_dir = '/home/jan/Documents/EPFL/project_2/data/ZENODO/images/'
forestloss_target_dir = '/home/jan/Documents/EPFL/project_2/data/ZENODO/deforestation_masks/'
for sampleid in tqdm.tqdm(xx):
    
    shutil.rmtree(os.path.join(img_target_dir, str(sampleid)))
    os.remove(os.path.join(forestloss_target_dir, str(sampleid) + ".tif"))
    


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 123/123 [00:00<00:00, 4652.69it/s]
